In [11]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import duckdb




In [43]:

##Phase 1: Data inspection... We want to know the data type, the number of missing values in the dataset, 
##and to have an idea of the dataset by printing the first 5 rows.
df=pd.read_csv(r"C:\Users\Usuari\Downloads\reporte_cruceros_revenue_management.csv")
print(df.head())
df.info()
df.describe()
df.isna().sum()

# # We describe the dataset; It contains 20 columns with different information
#Id re



      RES_ID Fecha_Viaje Fecha_Reserva  Lead_Time_Dias  Noches_Estancia  \
0  RES-00001  2018-01-05    2017-08-18             140                6   
1  RES-00002  2018-01-05    2017-09-14             113                6   
2  RES-00003  2018-01-05    2017-11-01              65                6   
3  RES-00004  2018-01-05    2017-10-31              66                6   
4  RES-00005  2018-01-05    2017-09-17             110                6   

              Barco      Compañia     Tipo_Ruta Suite_Type Booking_Source  \
0  MSC World Europa  MSC Cruceros  Mediterráneo    Balcony         Direct   
1  MSC World Europa  MSC Cruceros  Mediterráneo   Interior            B2B   
2  MSC World Europa  MSC Cruceros  Mediterráneo  Oceanview            Web   
3  MSC World Europa  MSC Cruceros  Mediterráneo   Interior            Web   
4  MSC World Europa  MSC Cruceros  Mediterráneo  Oceanview         Direct   

               Package Guest_Country  Cabinas_Totales_Barco  \
0        All Inclusive 

RES_ID                               0
Fecha_Viaje                          0
Fecha_Reserva                        0
Lead_Time_Dias                       0
Noches_Estancia                      0
Barco                                0
Compañia                             0
Tipo_Ruta                            0
Suite_Type                           0
Booking_Source                       0
Package                              0
Guest_Country                        0
Cabinas_Totales_Barco                0
Cabinas_Reservadas                   0
Porcentaje_Ocupacion_Ciclo           0
Pasajeros_Reserva                    0
Tripulacion                          0
Ingreso_Total_Reserva_USD            0
Gasto_Promedio_Diario_Huesped_USD    0
Puntuacion_Satisfaccion              0
dtype: int64

In [50]:
#Data cleaning. Format all the columns to the correct data type and delete duplicate rows 
df.drop_duplicates(inplace=True)
df['Fecha_Viaje']=pd.to_datetime(df['Fecha_Viaje'])
df['Fecha_Reserva']=pd.to_datetime(df['Fecha_Reserva'])
# CORRECTED VERSION
df = df[
    (df['Lead_Time_Dias'] > 0)
    & (df['Cabinas_Totales_Barco'] > 0)
    & (df['Cabinas_Reservadas'] > 0)
    & (df['Porcentaje_Ocupacion_Ciclo'] < 100)
    & (df['Pasajeros_Reserva'] > 0)  
    & (df['Tripulacion'] > 0)
    & (df['Puntuacion_Satisfaccion'] < 5.0)  
]
#We remove wrong data; check that all the columns have coherent data
df.describe()




,Fecha_Viaje,Fecha_Reserva,Lead_Time_Dias,Noches_Estancia,Cabinas_Totales_Barco,Cabinas_Reservadas,Porcentaje_Ocupacion_Ciclo,Pasajeros_Reserva,Tripulacion,Ingreso_Total_Reserva_USD,Gasto_Promedio_Diario_Huesped_USD,Puntuacion_Satisfaccion
count,62923,62923,62923.000000,62923.000000,62923.000000,62923.000000,62923.000000,62923.000000,62923.000000,62923.000000,62923.000000,62923.000000
mean,2020-01-20 07:31:30.024315392,2019-10-15 12:22:42.808511744,96.797769,7.773898,2124.909667,1.151725,90.130710,1.754653,1654.164979,3925.350613,167.103555,4.545738
min,2018-01-05 00:00:00,2017-07-15 00:00:00,5.000000,3.000000,1250.000000,1.000000,70.000000,1.000000,1253.000000,947.970000,98.430000,3.700000
25%,2019-01-13 00:00:00,2018-10-13 00:00:00,69.000000,6.000000,1643.000000,1.000000,86.150000,1.000000,1388.000000,2361.570000,140.120000,4.400000
50%,2020-01-24 00:00:00,2019-10-24 00:00:00,96.000000,7.000000,2612.000000,1.000000,90.840000,2.000000,1500.000000,3235.230000,163.410000,4.600000
75%,2021-01-30 00:00:00,2020-11-02 00:00:00,124.000000,9.000000,2732.000000,1.000000,94.910000,2.000000,2138.000000,4774.785000,190.920000,4.800000
max,2022-02-06 00:00:00,2022-02-01 00:00:00,267.000000,17.000000,2805.000000,2.000000,99.990000,6.000000,2350.000000,27348.300000,271.630000,4.900000
std,NaN,NaN,40.059345,2.770430,609.236554,0.358757,5.982874,0.855174,363.077527,2388.726983,31.927407,0.246417


In [59]:
#WE extract insights from data connecting SQL by Duck Library

#We extract general results from database: total revenues, the average spending by customer, and the average revenue
query_1_results= """ 
SELECT Barco,
COUNT() AS total_reservas,
SUM(Ingreso_Total_Reserva_USD) AS total_ingreso,
RANK() OVER(ORDER BY SUM(Ingreso_Total_Reserva_USD) DESC) AS ranking_revenues,
AVG(Ingreso_Total_Reserva_USD) AS ingreso_reserva,
AVG(Gasto_Promedio_Diario_Huesped_USD) AS gasto_persona,
AVG(Puntuacion_Satisfaccion) AS rate,
RANK() OVER(ORDER BY AVG(Puntuacion_Satisfaccion) ) AS ranking_rate,
FROM df 
GROUP BY Barco 
ORDER BY AVG(Puntuacion_Satisfaccion)

"""


query_2_route= """
SELECT 
Barco AS Ship,
Tipo_Ruta AS Route,
COUNT(*) AS Total_reservations,
AVG(Ingreso_Total_Reserva_USD) AS avg_revenue_bycustomer,
DENSE_RANK() OVER (ORDER BY AVG(Ingreso_Total_Reserva_USD) DESC) AS ranking_revenue
FROM df
GROUP BY Barco, Tipo_Ruta
"""


query_3_reservations="""
SELECT 
CASE
            WHEN Lead_Time_Dias<=30 THEN  'Last-hour'
            WHEN Lead_Time_Dias BETWEEN 31 AND 90 THEN 'Short-term'
            WHEN Lead_Time_Dias BETWEEN 90 AND 180 THEN 'Planned'
            WHEN Lead_Time_Dias> 180 THEN 'High-Anticipation' 
            END AS Lead_Time_Days,
COUNT(*) AS reservations_antelation,
AVG(Gasto_Promedio_Diario_Huesped_USD) AS avg_customerspend,
DENSE_RANK() OVER (ORDER BY AVG(Gasto_Promedio_Diario_Huesped_USD )DESC) AS ranking_customerspend, 
AVG( Ingreso_Total_Reserva_USD ) AS avg_customer_spend,
DENSE_RANK() OVER (ORDER BY AVG(Ingreso_Total_Reserva_USD)DESC) AS ranking_customerrevenue,
AVG(Puntuacion_Satisfaccion) AS avg_rating,
DENSE_RANK() OVER (ORDER BY AVG(Puntuacion_Satisfaccion) DESC) AS ranking_satisfaction
FROM df 
GROUP BY Lead_Time_Days
ORDER BY CASE  Lead_Time_Days
            WHEN  'Last-hour' THEN 1  
            WHEN  'Short-term' THEN 2
            WHEN  'Planned' THEN 3
            WHEN  'High-Anticipation'THEN 3 
END;

 
"""
query_4_seasonality="""
SELECT 
    CASE 
        WHEN EXTRACT (MONTH FROM Fecha_Viaje) BETWEEN 1 AND 3 THEN 'Winter'
        WHEN  EXTRACT (MONTH FROM Fecha_Viaje) BETWEEN 4 AND 9 THEN 'Summer High-Season'
        WHEN EXTRACT (MONTH FROM Fecha_Viaje) BETWEEN 9 AND 12 THEN  'Fall-Christmas'
END AS Season,
COUNT(*) AS  number_reservations,
SUM(Ingreso_Total_Reserva_USD) AS total_ingreso,
RANK() OVER(PARTITION BY Season ORDER BY SUM(Ingreso_Total_Reserva_USD) DESC) AS ranking_total_revenue,
AVG( Ingreso_Total_Reserva_USD ) AS avg_customer_spend,
DENSE_RANK() OVER (PARTITION BY Season ORDER BY AVG(Ingreso_Total_Reserva_USD) DESC) AS ranking_customerrevenue,
AVG(Puntuacion_Satisfaccion) AS rate,
RANK() OVER(PARTITION BY Season ORDER BY AVG(Puntuacion_Satisfaccion) DESC) AS ranking_rate
FROM df
GROUP BY Season
ORDER BY CASE  Season
            WHEN  'Winter' THEN 1  
            WHEN  'Summer High-season' THEN 2
            WHEN  'Fall-Christmas' THEN 3
END;

"""
df_view1=duckdb.query(query_1_results).df()
df_view2=duckdb.query(query_2_route).df()
df_view3=duckdb.query(query_3_reservations).df()
df_view4=duckdb.query(query_4_seasonality).df()
print(df_view4)
df_view1.to_csv('vista_barcos.csv', index=False)
df_view2.to_csv('vista_rutas.csv', index=False)
df_view3.to_csv('vista_lead_time.csv', index=False)

               Season  number_reservations  total_ingreso  \
0              Winter                18460   6.852616e+07   
1      Fall-Christmas                15417   5.955050e+07   
2  Summer High-Season                29046   1.189182e+08   

   ranking_total_revenue  avg_customer_spend  ranking_customerrevenue  \
0                      1         3712.143116                        1   
1                      1         3862.651886                        1   
2                      1         4094.132431                        1   

       rate  ranking_rate  
0  4.543624             1  
1  4.547045             1  
2  4.546388             1  
